# EECE 571Q Lecture 8: simulations with `stim`

This code is based on that provided in the [Stim tutorial here](https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb).

In [ ]:
# !pip install stim
# !pip install pymatching
# !pip install sinter

In [ ]:
import stim
import matplotlib.pyplot as plt

In [ ]:
circuit = stim.Circuit.generated(
    "color_code:memory_xyz",
    rounds=3,
    distance=3,
    before_round_data_depolarization=0.001
)

In [ ]:
print(circuit)

In [ ]:
circuit.diagram("timeline-svg")

In [ ]:
circuit.diagram("timeslice-svg")

In [ ]:
import numpy as np
import sinter

distances = [3, 5]

dep_error_rates = np.linspace(1e-6, 1e-4, 10)

# Source: adapted from https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb

tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated(
            "color_code:memory_xyz",
            rounds=d * 3,
            distance=d,
            before_round_data_depolarization=p,
        ),
        json_metadata={'d': d, 'p': p},
    )
    for d in distances
    for p in dep_error_rates
]

collected_stats = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=['pymatching'],
    max_shots=1_000_000,
    max_errors=500,
)

In [ ]:
# Source: adapted from https://github.com/quantumlib/Stim/blob/01f1aabd0fbc64d6943d42094a4bde96818399bd/doc/getting_started.ipynb
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: stats.json_metadata['d'],
)
ax.axline((0, 0), slope=1, label='p', linestyle='--', color='black')
ax.loglog()
ax.set_title("Colour Code Error Rates")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()